# Running magg from a JupyterHub

This notebook demonstrates how to use the `magg` Python API from a Jupyter
notebook, covering two deployment scenarios:

1. **Local processing** -- no AWS Lambda needed, runs on the hub itself
2. **Lambda processing** -- fans out to AWS Lambda for full-scale runs

Both paths use the same `agg()` function with different `backend` arguments.

## 1. Authentication

magg needs two sets of credentials:

- **NASA Earthdata** -- for reading source data (ICESat-2 HDF5 on NSIDC S3)
- **AWS** (optional) -- for writing output to S3 or invoking Lambda

Both use standard credential discovery -- if you already have `~/.netrc`
and `~/.aws/credentials` configured, skip this section entirely.

### Default credential locations

| Service | File | Format |
|---------|------|--------|
| Earthdata | `~/.netrc` | `machine urs.earthdata.nasa.gov login USERNAME password PASSWORD` |
| AWS | `~/.aws/credentials` | `[default]` profile with `aws_access_key_id` / `aws_secret_access_key` |

### Operator-managed hub (CryoCloud, Pangeo)

Credentials are pre-configured as environment variables or mounted files.
Nothing to do -- `earthaccess` and `boto3` discover them automatically.

### Override via environment variables

Only needed if the standard files above are not present:

In [1]:
import os

# Only needed if ~/.netrc is not configured:
# os.environ["EARTHDATA_USERNAME"] = "your_username"
# os.environ["EARTHDATA_PASSWORD"] = "your_password"

# Only needed if ~/.aws/credentials is not configured:
# os.environ["AWS_ACCESS_KEY_ID"] = "AKIA..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

## 2. Load a pipeline config

The config defines what data to read, how to aggregate it, and where to
write output. Use the built-in ATL06 config or load a custom YAML.

In [2]:
from magg import default_config, get_child_order

config = default_config("atl06")

print(f"Reader: {config.data_source['reader']}")
print(f"Groups: {config.data_source['groups']}")
print(f"Variables: {list(config.data_source['variables'].keys())}")
print(f"Child order: {get_child_order(config)}")

/home/espg/software/magg_deployment/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reader: h5coro
Groups: ['gt1l', 'gt1r', 'gt2l', 'gt2r', 'gt3l', 'gt3r']
Variables: ['h_li', 's_li']
Child order: 12


## 3. Build or load a catalog

The catalog maps morton cells to granule S3 URLs. Build one from CMR
(no credentials needed for the query itself) or load an existing one.

In [3]:
# Option A: Build a catalog (queries NASA CMR, takes ~30s)
!uv run python -m magg.catalog --cycle 22 --parent-order 6

Querying CMR for ATL06 v007
  Temporal: 2024-01-06T00:00:00Z,2024-04-06T23:59:59Z
  Total matching granules: 14710
  Retrieved 2000/14710...
  Retrieved 4000/14710...
  Retrieved 6000/14710...
  Retrieved 8000/14710...
  Retrieved 10000/14710...
  Retrieved 12000/14710...
  Retrieved 14000/14710...
  Retrieved 14710/14710...
Retrieved 14710 granules from CMR
Cell discovery: 1334 cells at order 6 from 27 parts
Pass 1: 1334 cells
Built 14710 granule polygons
Pass 2: 1330 cells with granule mappings

Timings:
  cell_discovery: 0.588s
  granule_polygons: 0.853s
  strtree_construction: 0.004s
  cell_polygons: 0.398s
  strtree_queries: 0.970s
  total: 2.814s

Catalog statistics:
  Parent cells: 1330
  Granules per cell: min=13, max=390, avg=57.6

Catalog saved to: catalog_ATL06_cycle22_order6.json
Total time: 62.8s


In [4]:
# Option B: Use an existing catalog
catalog_path = "catalog_ATL06_cycle22_order6.json"

import json
with open(catalog_path) as f:
    cat = json.load(f)
print(f"Cells: {cat['metadata']['total_cells']}")
print(f"Granules: {cat['metadata']['total_granules']}")

Cells: 1330
Granules: 14710


## 4. Local processing

Process cells locally using threads. No Lambda infrastructure needed.
Good for testing, small regions, or when you don't have AWS set up.

Use `driver="https"` to read data via HTTPS (works from anywhere).
Use `driver="s3"` for direct S3 access (faster, but only from us-west-2).

In [5]:
from magg import agg

# Dry run -- preview without processing
preview = agg(
    config,
    catalog=catalog_path,
    store="./test_output.zarr",
    driver="https",
    dry_run=True,
)
preview

{'dry_run': True,
 'total_cells': 1330,
 'granules_per_cell_min': 13,
 'granules_per_cell_max': 390,
 'granules_per_cell_avg': 57.57518796992481,
 'store_path': './test_output.zarr'}

In [6]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

# Process 2 cells locally, write to local Zarr
# driver="https" reads via HTTPS (works outside us-west-2)
results = agg(
    config,
    catalog=catalog_path,
    store="./test_output.zarr",
    driver="https",
    max_cells=2,
    max_workers=2,
    overwrite=True,
)

print(f"\nCells with data: {results['cells_with_data']}")
print(f"Total observations: {results['total_obs']:,}")
print(f"Wall time: {results['wall_time_s']:.1f}s")

Processing 2 of 1330 cells (local, 2 workers, driver=https)
You're now authenticated with NASA Earthdata Login
Using token with expiration date 04/19/2026
Processing morton cell: -6134112
Processing morton cell: -6132434
  Processing 23 granules from catalog
  Processing 19 granules from catalog
  Processed 19/19 files
  Read 57,153 observations
  Calculating statistics for order-12 cells...
  Statistics: 763/4096 cells with data
Completed morton -6132434 in 135.1s
  [1/2] -6132434: 57,153 obs
  Processed 23/23 files
  Read 110,693 observations
  Calculating statistics for order-12 cells...
  Statistics: 1147/4096 cells with data
Completed morton -6134112 in 215.1s
  [2/2] -6134112: 110,693 obs
/home/espg/software/magg_deployment/.venv/lib64/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.w


Cells with data: 2
Total observations: 167,846
Wall time: 215.2s


## 5. Inspect the output

The output is a Zarr v3 store following the DGGS convention.

In [7]:
import zarr

store = zarr.open_group("./test_output.zarr", mode="r")
child_order = get_child_order(config)
group = store[str(child_order)]

print(f"Arrays: {list(group.array_keys())}")
for name in ["count", "h_mean"]:
    arr = group[name]
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

Arrays: ['cell_ids', 'count', 'h_max', 'h_mean', 'h_min', 'h_q25', 'h_q50', 'h_q75', 'h_sigma', 'h_variance', 'morton']
  count: shape=(5447680,), dtype=int32
  h_mean: shape=(5447680,), dtype=float32


## 6. Lambda processing (optional)

For full-scale runs, fan out to AWS Lambda. Requires:
- AWS credentials with Lambda invoke + S3 write permissions
- A deployed Lambda function (see `deployment/` docs)

The API is identical -- just change `backend` and `store`.

In [8]:
# Uncomment to run on Lambda:

results = agg(
    config,
    catalog=catalog_path,
    store="s3://xagg/atl06/morton_aggregation.zarr",
    backend="lambda",
    max_cells=10,  # remove for full run
)

print(f"Cells: {results['cells_with_data']}")
print(f"Observations: {results['total_obs']:,}")
print(f"Wall time: {results['wall_time_s']:.1f}s")
print(f"Lambda compute: {results['lambda_time_s']:.1f}s")

Processing 10 of 1330 cells (lambda, 1700 workers)
Found credentials in shared credentials file: ~/.aws/credentials
Found credentials in shared credentials file: ~/.aws/credentials
Done: 0 cells, 0 obs, 10 errors, 6.2s


Cells: 0
Observations: 0
Wall time: 6.2s
Lambda compute: 0.0s


## 7. Custom configs

You can load a custom YAML config or modify the default programmatically.
See the [custom_aggregations](custom_aggregations.ipynb) notebook for
detailed examples of adding variables, changing statistics, etc.

In [ ]:
from magg import load_config

# Load from a custom YAML file:
# config = load_config("my_custom_config.yaml")

# Or set catalog/store in the config itself:
config.catalog = catalog_path
config.output["store"] = "./output.zarr"

# Then agg() uses them as defaults (CLI/kwargs still override):
# results = agg(config, max_cells=5)